# 17 — Seed variance

Every model comparison in the paper rests on one run per system, and the five
systems span two $F_1$ points. mBERT leads XLM-RoBERTa by 0.0071, which is
inside the range a different random seed can produce. The CRF finding rests on
an exact crossover at 400 instances with no exceptions, and a reviewer will ask
whether that survives reseeding.

This notebook trains two additional seeds for each transformer and reports mean
and standard deviation, then checks whether the per-type crossover holds.

**Designed to run unattended.** Every run is saved as it finishes and completed
runs are skipped on restart, so you can stop and resume. Budget about 23 GPU
hours for all six runs; the order is cheapest-first so partial results are still
useful.

**Run from the repository root.** Kernel: `Python (tka)`.

## Cell 1: Setup

In [1]:
from pathlib import Path
import json, sys, time, gc
from collections import defaultdict
import numpy as np
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Repo root: {ROOT}")

DATA = ROOT / "data" / "processed" / "bio_v2"
RES  = ROOT / "results" / "ner"
SEEDRES = RES / "seeds"
SEEDRES.mkdir(parents=True, exist_ok=True)

MODELS = {                       # cheapest first
    "mizbert": ("robzchhangte/MizBERT",            64),
    "xlmr":    ("xlm-roberta-base",                96),
    "mbert":   ("bert-base-multilingual-cased",    96),
}
SEEDS = [1, 2]                   # seed 42 is the run already reported

if not torch.cuda.is_available():
    sys.exit("No GPU")
pr = torch.cuda.get_device_properties(0)
VRAM = pr.total_memory / 1024**3
print(f"GPU: {pr.name}  {VRAM:.1f} GB")

splits = {s: json.load(open(DATA / f"mizo_ner_{s}.json", encoding="utf-8"))
          for s in ("train", "dev", "test")}
print({k: len(v) for k, v in splits.items()})

entity_types = ["EVENT","FAC","GPE","LANGUAGE","LAW","LOC",
                "NORP","ORG","PERSON","PRODUCT","WORK_OF_ART"]
tag_list = ["O"] + [f"{p}-{e}" for e in entity_types for p in ("B","I")]
tag2id = {t: i for i, t in enumerate(tag_list)}
id2tag = {i: t for t, i in tag2id.items()}

planned = [(m, s) for m in MODELS for s in SEEDS]
done = [f"{m}_seed{s}" for m, s in planned if (SEEDRES / f"{m}_seed{s}.json").exists()]
print(f"\nplanned runs: {len(planned)}   already done: {len(done)}")
for d in done: print("   skip", d)

Repo root: C:\Users\Haulai\mizo-ner
GPU: NVIDIA GeForce RTX 3060  12.0 GB
{'train': 352941, 'dev': 44118, 'test': 44118}

planned runs: 6   already done: 0


## Cell 2: Training routine

In [2]:
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          TrainingArguments, Trainer,
                          DataCollatorForTokenClassification, set_seed)
from seqeval.metrics import (classification_report, f1_score,
                             precision_score, recall_score)

class NERData(Dataset):
    def __init__(self, recs, tok, max_len):
        self.r, self.tok, self.n = recs, tok, max_len
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        rec = self.r[i]
        enc = self.tok(rec["tokens"], is_split_into_words=True, max_length=self.n,
                       padding="max_length", truncation=True, return_tensors="pt")
        wid, lab, prev = enc.word_ids(batch_index=0), [], None
        for w in wid:
            if w is None:      lab.append(-100)
            elif w != prev:    lab.append(tag2id[rec["tags"][w]])
            else:              lab.append(-100)
            prev = w
        out = {"input_ids": enc["input_ids"].squeeze(0),
               "attention_mask": enc["attention_mask"].squeeze(0),
               "labels": torch.tensor(lab)}
        if "token_type_ids" in enc:
            out["token_type_ids"] = enc["token_type_ids"].squeeze(0)
        return out

def metrics(p):
    logits, labels = p
    pred = np.argmax(logits, axis=2)
    T, P = [], []
    for pr_, lb in zip(pred, labels):
        t, q = [], []
        for a, b in zip(pr_, lb):
            if b != -100:
                t.append(id2tag[int(b)]); q.append(id2tag[int(a)])
        T.append(t); P.append(q)
    return {"precision": precision_score(T, P), "recall": recall_score(T, P),
            "f1": f1_score(T, P)}

BATCH, ACCUM = (64, 1) if VRAM >= 10 else (32, 2)

def run(key, seed):
    out = SEEDRES / f"{key}_seed{seed}.json"
    if out.exists():
        print(f"  skip {key} seed {seed}"); return json.load(open(out, encoding="utf-8"))
    name, max_len = MODELS[key]
    print(f"\n{'='*58}\n  {key}  seed {seed}\n{'='*58}", flush=True)
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForTokenClassification.from_pretrained(
        name, num_labels=len(tag_list), id2label=id2tag, label2id=tag2id)
    ds = {s: NERData(splits[s], tok, max_len) for s in splits}

    args = TrainingArguments(
        output_dir=str(ROOT / "models" / f"_seed_{key}_{seed}"),
        eval_strategy="epoch", save_strategy="no",
        learning_rate=2e-5, per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
        num_train_epochs=5, weight_decay=0.01, warmup_ratio=0.1, fp16=True,
        logging_steps=1000, dataloader_num_workers=0, report_to="none",
        seed=seed, data_seed=seed)

    tr = Trainer(model=model, args=args, train_dataset=ds["train"],
                 eval_dataset=ds["dev"], compute_metrics=metrics,
                 data_collator=DataCollatorForTokenClassification(tok))
    t0 = time.time(); tr.train(); hours = (time.time()-t0)/3600

    pred = tr.predict(ds["test"])
    pi = np.argmax(pred.predictions, axis=2)
    T, P = [], []
    for pr_, lb in zip(pi, pred.label_ids):
        t, q = [], []
        for a, b in zip(pr_, lb):
            if b != -100:
                t.append(id2tag[int(b)]); q.append(id2tag[int(a)])
        T.append(t); P.append(q)
    ft = [x for s in T for x in s]; fp = [x for s in P for x in s]
    ent = [i for i, x in enumerate(ft) if x != "O"]
    rep = classification_report(T, P, output_dict=True, digits=4)

    res = {"model": name, "seed": seed, "hours": round(hours, 2),
           "f1_micro": round(float(f1_score(T, P)), 4),
           "f1_macro": round(float(f1_score(T, P, average="macro")), 4),
           "token_accuracy_entity": round(float(np.mean(
               [ft[i] == fp[i] for i in ent]))*100, 2),
           "per_type": {k: round(v["f1-score"], 4) for k, v in rep.items()
                        if k not in ("micro avg","macro avg","weighted avg")}}
    with open(out, "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2)
    print(f"  {key} seed {seed}: micro {res['f1_micro']:.4f}  "
          f"macro {res['f1_macro']:.4f}  ({hours:.2f} h)")
    del model, tr; gc.collect(); torch.cuda.empty_cache()
    return res

print("run() defined")

run() defined


## Cell 3: Run everything

Safe to interrupt and re-run; completed runs are skipped. Cheapest model first,
so partial results are still usable.

In [3]:
t_all = time.time()
for key in MODELS:
    for seed in SEEDS:
        try:
            run(key, seed)
        except Exception as e:
            print(f"  !! {key} seed {seed} failed: {type(e).__name__}: {e}")
            gc.collect(); torch.cuda.empty_cache()
print(f"\ntotal {(time.time()-t_all)/3600:.2f} h")


  mizbert  seed 1


Some weights of BertForTokenClassification were not initialized from the model checkpoint at robzchhangte/MizBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.099100,0.088974,0.832239,0.839640,0.835923
2,0.076600,0.077370,0.845528,0.865074,0.855189
3,0.063100,0.072475,0.858011,0.879981,0.868857
4,0.052100,0.071015,0.865712,0.885415,0.875453
5,0.044300,0.072466,0.870853,0.886386,0.878551


  mizbert seed 1: micro 0.8802  macro 0.7306  (1.67 h)

  mizbert  seed 2


Some weights of BertForTokenClassification were not initialized from the model checkpoint at robzchhangte/MizBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.096900,0.088427,0.825062,0.848549,0.836641
2,0.076000,0.076609,0.849636,0.864052,0.856783
3,0.062900,0.070699,0.862649,0.879623,0.871053
4,0.052300,0.069910,0.869005,0.885927,0.877384
5,0.044500,0.071088,0.874847,0.889334,0.882031


  mbert seed 1: micro 0.8820  macro 0.7382  (2.44 h)

  mbert  seed 2


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.095400,0.087790,0.822280,0.851054,0.836420
2,0.074300,0.075614,0.847636,0.867579,0.857492
3,0.060900,0.070936,0.862154,0.881395,0.871668
4,0.052000,0.069671,0.868173,0.887681,0.877819
5,0.042800,0.071135,0.874050,0.889044,0.881483


  mbert seed 2: micro 0.8805  macro 0.7347  (9.41 h)

total 20.48 h


## Cell 4: Mean and standard deviation

In [4]:
SEED42 = {   # the runs already reported in the paper
 "xlmr":    {"f1_micro":0.8739,"f1_macro":0.7141,"token_accuracy_entity":87.75},
 "mizbert": {"f1_micro":0.8788,"f1_macro":0.7274,"token_accuracy_entity":88.27},
 "mbert":   {"f1_micro":0.8810,"f1_macro":0.7347,"token_accuracy_entity":88.44},
}
NAMES = {"xlmr":"XLM-RoBERTa base","mizbert":"MizBERT","mbert":"mBERT cased"}

runs = defaultdict(list)
for key in MODELS:
    runs[key].append(dict(SEED42[key], seed=42))
    for seed in SEEDS:
        p = SEEDRES / f"{key}_seed{seed}.json"
        if p.exists():
            runs[key].append(json.load(open(p, encoding="utf-8")))

print(f"{'Model':<20}{'n':>3}{'micro F1':>20}{'macro F1':>20}")
print("-" * 66)
summary = {}
for key in MODELS:
    r = runs[key]
    mi = np.array([x["f1_micro"] for x in r])
    ma = np.array([x["f1_macro"] for x in r])
    summary[key] = {"n": len(r),
                    "micro_mean": round(float(mi.mean()),4), "micro_sd": round(float(mi.std(ddof=1)),4) if len(r)>1 else None,
                    "macro_mean": round(float(ma.mean()),4), "macro_sd": round(float(ma.std(ddof=1)),4) if len(r)>1 else None,
                    "micro_all": [float(x) for x in mi], "macro_all": [float(x) for x in ma]}
    s1 = f"{mi.mean():.4f} ± {mi.std(ddof=1):.4f}" if len(r)>1 else f"{mi.mean():.4f}"
    s2 = f"{ma.mean():.4f} ± {ma.std(ddof=1):.4f}" if len(r)>1 else f"{ma.mean():.4f}"
    print(f"{NAMES[key]:<20}{len(r):>3}{s1:>20}{s2:>20}")

print("\nindividual runs (micro):")
for key in MODELS:
    print(f"  {NAMES[key]:<20}" + "  ".join(f"{x:.4f}" for x in summary[key]["micro_all"]))

Model                 n            micro F1            macro F1
------------------------------------------------------------------
MizBERT               3     0.8789 ± 0.0012     0.7281 ± 0.0023
XLM-RoBERTa base      3     0.8725 ± 0.0012     0.7115 ± 0.0023
mBERT cased           3     0.8812 ± 0.0008     0.7359 ± 0.0020

individual runs (micro):
  MizBERT             0.8788  0.8802  0.8778
  XLM-RoBERTa base    0.8739  0.8716  0.8720
  mBERT cased         0.8810  0.8820  0.8805


## Cell 5: Does the ranking survive?

The paper claims mBERT $>$ MizBERT $>$ XLM-RoBERTa. If the spread between models
is comparable to the spread within a model across seeds, that ordering is not
supportable and the paper should say all three are equivalent.

In [5]:
order = sorted(MODELS, key=lambda k: -summary[k]["micro_mean"])
print("ranking by mean micro-F1:", " > ".join(NAMES[k] for k in order))

sds = [summary[k]["micro_sd"] for k in MODELS if summary[k]["micro_sd"] is not None]
if sds:
    pooled = float(np.mean(sds))
    spread = summary[order[0]]["micro_mean"] - summary[order[-1]]["micro_mean"]
    print(f"\nbetween-model spread (best - worst): {spread:.4f}")
    print(f"mean within-model SD across seeds  : {pooled:.4f}")
    print(f"ratio: {spread/pooled:.2f}")
    if spread < 2*pooled:
        print("\nThe spread between models is within twice the seed noise.")
        print("Report the three transformers as equivalent, not ranked.")
    else:
        print("\nThe spread exceeds twice the seed noise; the ordering holds.")

    print("\npairwise gaps against seed noise:")
    for i in range(len(order)):
        for j in range(i+1, len(order)):
            a, b = order[i], order[j]
            d = summary[a]["micro_mean"] - summary[b]["micro_mean"]
            print(f"  {NAMES[a]:<18} - {NAMES[b]:<18} {d:+.4f}"
                  f"   {'within' if abs(d) < pooled else 'beyond'} 1 SD")
else:
    print("\nOnly one seed per model so far; run Cell 3.")

ranking by mean micro-F1: mBERT cased > MizBERT > XLM-RoBERTa base

between-model spread (best - worst): 0.0087
mean within-model SD across seeds  : 0.0011
ratio: 8.16

The spread exceeds twice the seed noise; the ordering holds.

pairwise gaps against seed noise:
  mBERT cased        - MizBERT            +0.0023   beyond 1 SD
  mBERT cased        - XLM-RoBERTa base   +0.0087   beyond 1 SD
  MizBERT            - XLM-RoBERTa base   +0.0064   beyond 1 SD


## Cell 6: Does the CRF crossover hold?

The paper claims the CRF beats every transformer on types below 400 instances
and loses on every type above 450. That must hold across seeds, not just for
seed 42.

In [6]:
crf_p = RES / "baseline_crf.json"
xl_p  = RES / "evaluation_v2.json"
if not (crf_p.exists() and xl_p.exists()):
    print("run notebooks 04 and 16 first")
else:
    crf = json.load(open(crf_p, encoding="utf-8"))["per_type"]
    sup = {k: v["support"] for k, v in
           json.load(open(xl_p, encoding="utf-8"))["per_type"].items()}

    print(f"{'Type':<14}{'support':>9}{'CRF':>9}{'best transformer (any seed)':>30}{'':>4}")
    print("-" * 68)
    viol = []
    for t in sorted(sup, key=lambda k: -sup[k]):
        c = crf.get(t, {}).get("f1", 0.0)
        best = 0.0
        for key in MODELS:
            for r in runs[key]:
                best = max(best, r.get("per_type", {}).get(t, 0.0))
        if best == 0.0:
            continue
        crf_wins = c > best
        expect  = sup[t] < 400
        flag = "" if crf_wins == expect else "  <-- breaks the pattern"
        if flag: viol.append(t)
        print(f"{t:<14}{sup[t]:>9,}{c:>9.4f}{best:>26.4f}{flag}")

    print()
    if viol:
        print(f"Crossover does NOT hold cleanly across seeds; exceptions: {viol}")
        print("Soften the claim in Section 7.4 to a tendency rather than an exact split.")
    else:
        print("Crossover holds against the best transformer run at every seed.")

Type            support      CRF   best transformer (any seed)    
--------------------------------------------------------------------
PERSON           32,629   0.9095                    0.9216
GPE              11,495   0.8710                    0.8888
ORG              10,196   0.7888                    0.8084
NORP              1,942   0.7881                    0.8027
LOC                 700   0.6950                    0.7204
LANGUAGE            479   0.7645                    0.8008
WORK_OF_ART         381   0.8509                    0.8444
FAC                 327   0.6736                    0.6246
PRODUCT             291   0.5304                    0.4561
EVENT                90   0.6087                    0.5729
LAW                  68   0.7742                    0.7376

Crossover holds against the best transformer run at every seed.


## Cell 7: Save and emit LaTeX

In [7]:
out = {"seeds_per_model": {k: summary[k]["n"] for k in MODELS},
       "summary": summary,
       "note": "seed 42 is the run reported in the main tables"}
with open(RES / "seed_variance.json", "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2)
print("-> results/ner/seed_variance.json\n")

print("% ---- Table: model comparison with seed variance ----")
for key in order:
    d = summary[key]
    if d["micro_sd"] is not None:
        print(f"{NAMES[key]:<20}& {d['micro_mean']:.4f} $\\pm$ {d['micro_sd']:.4f} "
              f"& {d['macro_mean']:.4f} $\\pm$ {d['macro_sd']:.4f} & {d['n']} \\\\")
    else:
        print(f"{NAMES[key]:<20}& {d['micro_mean']:.4f} & {d['macro_mean']:.4f} & 1 \\\\")

-> results/ner/seed_variance.json

% ---- Table: model comparison with seed variance ----
mBERT cased         & 0.8812 $\pm$ 0.0008 & 0.7359 $\pm$ 0.0020 & 3 \\
MizBERT             & 0.8789 $\pm$ 0.0012 & 0.7281 $\pm$ 0.0023 & 3 \\
XLM-RoBERTa base    & 0.8725 $\pm$ 0.0012 & 0.7115 $\pm$ 0.0023 & 3 \\
